In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from harness.data import load_sample
from harness.models import load_model, run_classification
from harness.metrics import score_predictions

HF_ID = "Qwen/Qwen2.5-1.5B-Instruct"
PROMPT_SST2 = (
    "Classify the sentiment of the following text. "
    "Answer with exactly one word: positive or negative. "
    "Do not add any explanation.\n\nText: {text}\n\nAnswer:"
)

ds = load_sample("stanfordnlp/sst2", "validation", n_samples=200, seed=42)
tok, model = load_model(HF_ID)
rows, stats = run_classification(
    tok, model, ds, PROMPT_SST2, text_key="sentence", detokenize=True
)

print("pred counts:", {k: sum(r["pred"] == k for r in rows) for k in (0, 1, None)})
result = {"model": HF_ID, "task": "sst2_validation", **score_predictions(rows), **stats}
print(pd.DataFrame([result]).to_string(index=False))